In [1]:
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

df = pd.read_csv("../data/processed/clean_reviews.csv")
df.head()

,text,deceptive,label,clean_text
0,We stayed for a one night getaway with family ...,truthful,0,we stayed for a one night getaway with family ...
1,Triple A rate with upgrade to view room was le...,truthful,0,triple a rate with upgrade to view room was le...
2,This comes a little late as I'm finally catchi...,truthful,0,this comes a little late as i m finally catchi...
3,The Omni Chicago really delivers on all fronts...,truthful,0,the omni chicago really delivers on all fronts...
4,I asked for a high floor away from the elevato...,truthful,0,i asked for a high floor away from the elevato...


In [2]:
X = df["clean_text"]
y = df["label"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (1596,)
y shape: (1596,)


In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("\ny_train dağılımı:")
print(y_train.value_counts())
print("\ny_test dağılımı:")
print(y_test.value_counts())

X_train shape: (1276,)
X_test shape: (320,)

y_train dağılımı:
label
1    640
0    636
Name: count, dtype: int64

y_test dağılımı:
label
1    160
0    160
Name: count, dtype: int64


In [4]:
tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=5000,
    ngram_range=(1, 2)
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("X_train_tfidf shape:", X_train_tfidf.shape)
print("X_test_tfidf shape:", X_test_tfidf.shape)

X_train_tfidf shape: (1276, 5000)
X_test_tfidf shape: (320, 5000)


In [5]:
model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train_tfidf, y_train)

print("Model eğitildi.")

Model eğitildi.


In [6]:
y_pred = model.predict(X_test_tfidf)

print("İlk 10 tahmin:")
print(y_pred[:10])

İlk 10 tahmin:
[1 1 1 0 1 0 0 0 0 0]


In [7]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.890625

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.86      0.89       160
           1       0.87      0.92      0.89       160

    accuracy                           0.89       320
   macro avg       0.89      0.89      0.89       320
weighted avg       0.89      0.89      0.89       320


Confusion Matrix:
[[138  22]
 [ 13 147]]


In [8]:
import joblib

joblib.dump(model, "../results/model_logreg.pkl")
joblib.dump(tfidf, "../results/tfidf_vectorizer.pkl")
joblib.dump((X_test, y_test, y_pred), "../results/test_outputs.pkl")

print("Model, vectorizer ve test çıktıları kaydedildi.")

Model, vectorizer ve test çıktıları kaydedildi.


In [10]:
import os

os.makedirs("../results/metrics", exist_ok=True)
print("results/metrics klasörü hazır.")

results/metrics klasörü hazır.


In [11]:
report = classification_report(y_test, y_pred)
accuracy = accuracy_score(y_test, y_pred)

with open("../results/metrics/classification_report.txt", "w") as f:
    f.write(f"Accuracy: {accuracy}\n\n")
    f.write(report)

print("Classification report kaydedildi.")

Classification report kaydedildi.
